In [2]:
import numpy as np
from sklearn.decomposition import FastICA

S=np.array([[1,1],[2,-1],[3,1],[4,-1]])
#Mix them
A=np.array([[1,2],[3,4]])
X = S@A.T
ica = FastICA(n_components=2,random_state=0)
U=ica.fit_transform(X)
print("Origina Sources :\n",S)
print("Recovered Sorurces (approx):\n",U)

Origina Sources :
 [[ 1  1]
 [ 2 -1]
 [ 3  1]
 [ 4 -1]]
Recovered Sorurces (approx):
 [[-1.  1.]
 [-1. -1.]
 [ 1.  1.]
 [ 1. -1.]]


In [3]:
from sklearn.neural_network import MLPRegressor

X=np.array([[1,2,3,4],[2,3,4,5],[3,4,5,6],[4,5,6,7]],dtype=float)

autoencoders = MLPRegressor(hidden_layer_sizes=(2,),activation="relu",max_iter=5000,random_state=0)

autoencoders.fit(X,X)

X_reconstructed = autoencoders.predict(X)

W=autoencoders.coefs_[0]
b=autoencoders.intercepts_[0]
hidden_2d = np.maximum(0,X@W+b)

print("Hidden 4 features ",X)
print("\nHidden (2 Features) ",np.round(hidden_2d,2))
print("\nReconstruction :\n",np.round(X_reconstructed,2))


Hidden 4 features  [[1. 2. 3. 4.]
 [2. 3. 4. 5.]
 [3. 4. 5. 6.]
 [4. 5. 6. 7.]]

Hidden (2 Features)  [[4.35 1.56]
 [5.81 2.11]
 [7.27 2.66]
 [8.74 3.21]]

Reconstruction :
 [[2.05 2.76 3.4  3.99]
 [2.52 3.33 4.09 5.  ]
 [2.99 3.89 4.78 6.  ]
 [3.45 4.45 5.47 7.  ]]


In [4]:
#Implementation of active learning
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

#Create simple dataset
X,y=make_classification(n_samples=100,n_features=2,n_redundant=0,n_informative=2,random_state=0)

labeled=np.arange(10)
unlabeled = np.arange(10,100)

model=LogisticRegression(solver="liblinear")

for i in range(5):
    model.fit(X[labeled],y[labeled])
    probs = model.predict_proba(X[unlabeled])[:,1]

    query=np.argmin(np.abs(probs-0.5))

    labeled =np.append(labeled,unlabeled[query])
    unlabeled = np.delete(unlabeled,query)
    print("iteration ",i+1," | labeled size ",len(labeled))

iteration  1  | labeled size  11
iteration  2  | labeled size  12
iteration  3  | labeled size  13
iteration  4  | labeled size  14
iteration  5  | labeled size  15


In [7]:

## to classify data by computing wiighted

# Simple 1D input data
X=np.array([[1],[2],[3],[4]])
y=np.array([0,0,1,1])

centers = np.array([[1],[4]])
sigma = 1.0

def rbf(x,c):
    return np.exp(-np.linalg.norm(x-c)**2/(2*sigma**2))
H = np.array([[rbf(x,c) for c in centers] for x in X])

w=np.linalg.pinv(H) @ y

y_pred = H@w
y_pred_class = (y_pred>0.5).astype(int)

print("Predicted ",y_pred_class)

Predicted  [0 0 1 1]


In [7]:
# Implementations of ECLAT algorithim

transactions = [{"A","B"},{"A","C"},{"B","C"},{"A","B"}];

min_sup=2

tid={}
for i,t in enumerate(transactions):
    for item in t:
        if item not in tid:
            tid[item]=set()
        tid[item].add(i)    
print("tid lists ",tid)      


# Step 2 : Build tid list
AB=tid["A"] & tid["B"] # intersection
print("Support of (A,B): ",len(AB))

# check if frequent
if len(AB)>=min_sup:
    print("AB is frequent")


tid lists  {'B': {0, 2, 3}, 'A': {0, 1, 3}, 'C': {1, 2}}
Support of (A,B):  2
AB is frequent


In [12]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

X,y=make_classification(n_samples=500,n_features=5,random_state=0)
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.3,random_state=0)
bag=BaggingClassifier(estimator=DecisionTreeClassifier(),n_estimators=20,bootstrap=True,random_state=0)
bag.fit(Xtr,ytr)
print("Bagging accuracy ",accuracy_score(yte,bag.predict(Xte)))

Bagging accuracy  0.9133333333333333


In [14]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score

X,y=make_classification(n_samples=500,n_features=5,random_state=0)
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.3,random_state=0)
ada=AdaBoostClassifier(estimator=DecisionTreeClassifier(),n_estimators=50,learning_rate=1.0,random_state=0)
ada.fit(Xtr,ytr)
print("Bagging accuracy ",accuracy_score(yte,ada.predict(Xte)))

Bagging accuracy  0.9066666666666666


In [19]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

gbm=GradientBoostingClassifier(n_estimators=100,learning_rate=0.1,random_state=0)
gbm.fit(Xtr,ytr)
print("GBM accuracy ",accuracy_score(yte,gbm.predict(Xte)))


GBM accuracy  0.9266666666666666


In [21]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error

X,y=make_regression(n_samples=200,n_features=5,noise=10,random_state=0)
model = ElasticNet(alpha=1.0,l1_ratio=0.5)
# alpha regularization strengeth
# l1_ratio=balance

model.fit(Xtr,ytr)
y_pred=model.predict(Xte)

print("MSE ",mean_squared_error(yte,y_pred))

MSE  0.25000816326530617
